# Terminal‑Bench 2.0 demo (Trace‑Bench)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgentOpt/Trace-Bench/blob/main/notebooks/05_terminal_bench_demo.ipynb)

This notebook demonstrates how to run **Terminal‑Bench 2.0** tasks via **Harbor** from Trace‑Bench.

Key points:
- Tasks are referenced as `terminal_bench:<task_slug>` (e.g. `terminal_bench:regex-log`).
- In Colab, Terminal‑Bench typically requires **Daytona** (`DAYTONA_API_KEY`) because Docker is not available.
- The optimization target is the **Terminus‑2 prompt template** (a trainable Trace node).


In [ ]:
# If running from a cloned repo, install Trace-Bench in editable mode.
!pip -q install -e .


In [ ]:
import os
from pathlib import Path

# Pick a persistent runs directory.
# In Colab you likely want to mount Drive first and set this under /content/drive/...
RUNS_DIR = Path(os.environ.get('TRACE_BENCH_RUNS_DIR', 'runs')).resolve()
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print('RUNS_DIR =', RUNS_DIR)


In [ ]:
# List available trainers (from Trace/OpenTrace) and the curated TB task subset.
!trace-bench list-trainers | head -n 40
print('\n--- terminal_bench sample tasks ---')
!trace-bench list-tasks --bench terminal_bench --root .


## 1) Stub smoke (offline)

The shipped config `configs/terminal_bench_demo.yaml` defaults to `mode: stub`, meaning Harbor is **not** invoked.
This is useful for CI and quick sanity checks.


In [ ]:
!trace-bench validate --config configs/terminal_bench_demo.yaml --runs-dir "$RUNS_DIR" --strict
!trace-bench run --config configs/terminal_bench_demo.yaml --runs-dir "$RUNS_DIR"


## 2) Real mode (Harbor + Daytona)

Requirements:
- `harbor` CLI available (`pip install harbor` typically provides it)
- `DAYTONA_API_KEY` exported (Colab recommended)
- A Harbor model name provided via `HARBOR_MODEL` (e.g. `openai/gpt-4o-mini`)

If you don't have these, keep using stub mode.


In [ ]:
import shutil, textwrap

if shutil.which('harbor') is None:
    print('harbor CLI not found. Install it first (e.g. pip install harbor).')
else:
    print('harbor CLI found:', shutil.which('harbor'))

print('DAYTONA_API_KEY set:', bool(os.environ.get('DAYTONA_API_KEY')))
print('HARBOR_MODEL:', os.environ.get('HARBOR_MODEL'))


In [ ]:
# Create a small real-mode config on the fly.
# NOTE: replace trainer settings / timeouts as needed.
real_cfg = f"""
runs_dir: {RUNS_DIR}
mode: real
seeds: [123]
max_workers: 1
fail_fast: false

tasks:
  - id: terminal_bench:regex-log
    eval_kwargs:
      harbor_dataset: terminal-bench@2.0
      harbor_env: daytona
      # read from env in the guide if omitted
      harbor_model: null
      harbor_cli_timeout_seconds: 1800

trainers:
  - id: PrioritySearch
    params_variants:
      - threads: 1
        ps_steps: 1
        ps_batches: 1
        ps_candidates: 2
        ps_proposals: 2
        ps_mem_update: 1
"""

cfg_path = Path('configs/terminal_bench_real_tmp.yaml')
cfg_path.write_text(textwrap.dedent(real_cfg))
print('Wrote', cfg_path)
print(cfg_path.read_text()[:400])


In [ ]:
# Run a single TB task in real mode.
# This will write Harbor artifacts under runs/<run_id>/jobs/<job_id>/artifacts/terminal_bench/.
!trace-bench validate --config configs/terminal_bench_real_tmp.yaml --runs-dir "$RUNS_DIR" --strict
!trace-bench run --config configs/terminal_bench_real_tmp.yaml --runs-dir "$RUNS_DIR"
